# 🧪 GIAI ĐOẠN 3 (NHÓM 2): TRANSFORMER ABLATION STUDY & SO SÁNH 3 BỘ TOKENIZER
**Đề tài 8: Dịch máy Anh – Việt (IWSLT'15 En–Vi) | Môn học: Học sâu**  
**Phụ trách: Sơn (Phần 3 – Nhóm trưởng)**  
---
> **Lưu ý:** Chọn GPU trước khi chạy: **Runtime -> Change runtime type -> T4 GPU** và bật công tắc Colab Secret **`BTL_DL_COLAB`**.
>
> **Đã hoàn thành trước đó (Mốc Base):** `transformer_bpe_base` (`N=4, H=8, use_pe=True, smoothing=0.1`) — Greedy `25.76` / Beam(k=4) **`26.71 BLEU`**.
>
> Notebook này chạy **6 cấu hình còn lại** (mỗi ô tự động: **Train -> Eval Greedy & Beam(k=4) -> Lưu ngay lên Google Drive** để không mất dữ liệu nếu Colab ngắt):
> 1. **Ablation 2 (`L2H8`):** Giảm số tầng `N=2, H=8`
> 2. **Ablation 3 (`L4H4`):** Giảm số head `N=4, H=4`
> 3. **Ablation 4 (`noPE`):** Tắt Positional Encoding (`use_pe=False`)
> 4. **Ablation 5 (`noLS`):** Tắt Label Smoothing (`smoothing=0.0`)
> 5. **Tokenizer Word:** `transformer_word_base`
> 6. **Tokenizer SentencePiece:** `transformer_spm_base`

### Bước 1: Kiểm tra GPU & Mount Google Drive chung

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

### Bước 2: Clone / Cập nhật mã nguồn nhánh `main` & Cài đặt thư viện

In [ ]:
import os

REPO_URL = "https://github.com/hoang180/Machine_Translation_Project.git"
PROJECT_DIR = "/content/Machine_Translation_Project"
BRANCH = "main"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("✓ Đã sẵn sàng mã nguồn nhánh main và thư viện!")

### Bước 3: Copy dữ liệu `data/processed/` & 3 bộ Tokenizer từ Google Drive chung (`BTL_DL_Translation`)

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"

!mkdir -p data/processed custom_tokenizers/word custom_tokenizers/bpe custom_tokenizers/spm checkpoints
!cp -r "{DRIVE_ROOT}/data/processed/." data/processed/
!cp -r "{DRIVE_ROOT}/custom_tokenizers/." custom_tokenizers/

!ls -lh data/processed/ custom_tokenizers/word/ custom_tokenizers/bpe/ custom_tokenizers/spm/

### Bước 4: Auto-login Weights & Biases (`btl-dl-translation-iwslt15`)

In [ ]:
import os
import wandb
from google.colab import userdata

api_key = userdata.get("BTL_DL_COLAB")
if api_key:
    os.environ["WANDB_API_KEY"] = api_key
    wandb.login(key=api_key, relogin=True)
else:
    wandb.login(relogin=True)

api = wandb.Api()
print("✓ Đang đăng nhập W&B:", api.viewer.username, "| Default entity:", api.default_entity)

### Bước 5: Định nghĩa hàm chạy trọn gói (`Train -> Eval Greedy & Beam(k=4) -> Lưu Drive`)
Hàm này tự động:
- Kiểm tra nếu run đã có `eval_result.json` trên Drive thì bỏ qua (tránh chạy lặp khi nối phiên Colab).
- Chạy `train_transformer.py` (ghi log trực tiếp lên W&B).
- Chạy `eval.py` với `beam_size=1` và `beam_size=4, alpha=0.6`, tự trích xuất điểm SacreBLEU lưu vào `eval_result.json`.
- Sao lưu ngay toàn bộ thư mục checkpoint (`best_model.pt`, `train_summary.json`, `eval_result.json`) lên Google Drive.

In [ ]:
import json
import os
import re
import shutil
import subprocess

DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"


def _run_and_stream(cmd: list[str]) -> str:
    print("\n>>>", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"Lệnh thất bại (exit code {ret}): {' '.join(cmd)}")
    return "".join(lines)


def _parse_bleu(output: str) -> dict:
    res = {}
    m = re.search(r"Điểm BLEU:\s*([0-9.]+)", output)
    if m:
        res["bleu"] = float(m.group(1))
    m_bp = re.search(r"Brevity Penalty:\s*([0-9.]+)", output)
    if m_bp:
        res["bp"] = float(m_bp.group(1))
    m_sig = re.search(r"Signature:\s*(.+)", output)
    if m_sig:
        res["signature"] = m_sig.group(1).strip()
    return res


def run_experiment(
    exp_dir_name: str,
    tokenizer: str = "bpe",
    num_layers: int = 4,
    num_heads: int = 8,
    use_pe: bool = True,
    smoothing: float = 0.1,
    epochs: int = 20,
    batch_size: int = 128,
    skip_if_done: bool = True,
):
    local_dir = os.path.join("checkpoints", exp_dir_name)
    drive_dir = os.path.join(DRIVE_ROOT, "checkpoints", exp_dir_name)
    drive_eval_file = os.path.join(drive_dir, "eval_result.json")

    if skip_if_done and os.path.exists(drive_eval_file):
        print(f"[SKIP] {exp_dir_name} đã hoàn thành và có sẵn trên Drive: {drive_eval_file}")
        with open(drive_eval_file, "r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), ensure_ascii=False, indent=2))
        return

    os.makedirs(local_dir, exist_ok=True)

    # 1. Train Transformer (có log W&B trực tiếp)
    train_cmd = [
        "python3", "train_transformer.py",
        "--config", "configs/transformer_config.yaml",
        "--tokenizer", tokenizer,
        "--num_layers", str(num_layers),
        "--num_heads", str(num_heads),
        "--use_pe", "true" if use_pe else "false",
        "--smoothing", str(smoothing),
        "--epochs", str(epochs),
        "--batch_size", str(batch_size),
        "--save_dir", local_dir,
    ]
    _run_and_stream(train_cmd)

    ckpt_path = os.path.join(local_dir, "best_model.pt")

    # 2. Eval Greedy (k=1) và Beam Search (k=4, alpha=0.6) trên tst2013
    out_greedy = _run_and_stream([
        "python3", "eval.py", "--checkpoint", ckpt_path, "--beam_size", "1"
    ])
    out_beam4 = _run_and_stream([
        "python3", "eval.py", "--checkpoint", ckpt_path, "--beam_size", "4", "--alpha", "0.6"
    ])

    eval_result = {
        "exp_dir": exp_dir_name,
        "tokenizer": tokenizer,
        "num_layers": num_layers,
        "num_heads": num_heads,
        "use_pe": use_pe,
        "smoothing": smoothing,
        "greedy": _parse_bleu(out_greedy),
        "beam4": _parse_bleu(out_beam4),
    }
    with open(os.path.join(local_dir, "eval_result.json"), "w", encoding="utf-8") as f:
        json.dump(eval_result, f, ensure_ascii=False, indent=2)

    # 3. Sao lưu ngay lên Google Drive
    os.makedirs(drive_dir, exist_ok=True)
    for fname in ("best_model.pt", "train_summary.json", "eval_result.json"):
        src_f = os.path.join(local_dir, fname)
        if os.path.exists(src_f):
            shutil.copy2(src_f, os.path.join(drive_dir, fname))

    print(f"\n✓ HOÀN TẤT & ĐÃ LƯU DRIVE: {drive_dir}")
    print(f"  -> Greedy BLEU: {eval_result['greedy'].get('bleu')} | Beam(k=4) BLEU: {eval_result['beam4'].get('bleu')}")

print("✓ Đã sẵn sàng hàm run_experiment()!")

---
## PHẦN A: 4 BIẾN THỂ ABLATION STUDY (TRÊN BPE TOKENIZER)

### Run 1 — Ablation 2: Giảm số tầng (`N=2, H=8`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_L2H8",
    tokenizer="bpe",
    num_layers=2,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=20,
    batch_size=128,
)

### Run 2 — Ablation 3: Giảm số head (`N=4, H=4`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_L4H4",
    tokenizer="bpe",
    num_layers=4,
    num_heads=4,
    use_pe=True,
    smoothing=0.1,
    epochs=20,
    batch_size=128,
)

### Run 3 — Ablation 4: Tắt Positional Encoding (`use_pe=False`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_noPE",
    tokenizer="bpe",
    num_layers=4,
    num_heads=8,
    use_pe=False,
    smoothing=0.1,
    epochs=20,
    batch_size=128,
)

### Run 4 — Ablation 5: Tắt Label Smoothing (`smoothing=0.0`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_bpe_noLS",
    tokenizer="bpe",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.0,
    epochs=20,
    batch_size=128,
)

---
## PHẦN B: SO SÁNH 3 BỘ TOKENIZER TRÊN TRANSFORMER (`Word` & `SentencePiece`)

### Run 5 — Transformer với Word Tokenizer (`transformer_word_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_word_base",
    tokenizer="word",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=20,
    batch_size=128,
)

### Run 6 — Transformer với SentencePiece Tokenizer (`transformer_spm_base`)

In [ ]:
run_experiment(
    exp_dir_name="transformer_spm_base",
    tokenizer="spm",
    num_layers=4,
    num_heads=8,
    use_pe=True,
    smoothing=0.1,
    epochs=20,
    batch_size=128,
)

---
## PHẦN C: TỔNG HỢP TOÀN BỘ BẢNG KẾT QUẢ CSV & JSON (CHÈN VÀO BÁO CÁO ACL)

In [ ]:
import csv
import json
import os
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/BTL_DL_Translation"
CKPT_ROOT = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

EXPERIMENTS = [
    ("transformer_bpe_base",  "Base (BPE, N=4, H=8, PE, LS=0.1)"),
    ("transformer_bpe_L2H8",  "Ablation: N=2 layers (H=8)"),
    ("transformer_bpe_L4H4",  "Ablation: H=4 heads (N=4)"),
    ("transformer_bpe_noPE",  "Ablation: No Positional Encoding"),
    ("transformer_bpe_noLS",  "Ablation: No Label Smoothing (0.0)"),
    ("transformer_word_base", "Tokenizer: Word-level (N=4, H=8)"),
    ("transformer_spm_base",  "Tokenizer: SentencePiece (N=4, H=8)"),
]

# Đảm bảo transformer_bpe_base (đã chạy ở notebook 02) có eval_result.json
base_eval_path = os.path.join(CKPT_ROOT, "transformer_bpe_base", "eval_result.json")
if not os.path.exists(base_eval_path) and os.path.exists(os.path.join(CKPT_ROOT, "transformer_bpe_base")):
    with open(base_eval_path, "w", encoding="utf-8") as f:
        json.dump({
            "exp_dir": "transformer_bpe_base",
            "tokenizer": "bpe",
            "num_layers": 4,
            "num_heads": 8,
            "use_pe": True,
            "smoothing": 0.1,
            "greedy": {"bleu": 25.76},
            "beam4": {"bleu": 26.71, "bp": 0.9429},
        }, f, ensure_ascii=False, indent=2)

rows = []
for folder, label in EXPERIMENTS:
    s_path = os.path.join(CKPT_ROOT, folder, "train_summary.json")
    e_path = os.path.join(CKPT_ROOT, folder, "eval_result.json")
    if not os.path.exists(s_path):
        continue
    with open(s_path, "r", encoding="utf-8") as f:
        s = json.load(f)
    ev = json.load(open(e_path, "r", encoding="utf-8")) if os.path.exists(e_path) else {}
    m_cfg = s.get("config", {}).get("model", {})
    t_cfg = s.get("config", {}).get("training", {})
    best_ep = min(s.get("history", []), key=lambda r: r["val/loss"], default={})
    rows.append({
        "Config": label,
        "Run": s.get("run_name", folder),
        "Tokenizer": s.get("tokenizer_type"),
        "Layers": m_cfg.get("num_layers"),
        "Heads": m_cfg.get("num_heads"),
        "PE": m_cfg.get("use_pe"),
        "LS": t_cfg.get("label_smoothing"),
        "Params": s.get("trainable_params"),
        "Avg_Epoch_s": s.get("avg_epoch_time_sec"),
        "Best_Epoch": best_ep.get("epoch"),
        "Val_Loss": s.get("best_val_loss"),
        "Val_PPL": best_ep.get("val/perplexity"),
        "Test_BLEU_Greedy": ev.get("greedy", {}).get("bleu"),
        "Test_BLEU_Beam4": ev.get("beam4", {}).get("bleu"),
    })

df = pd.DataFrame(rows)
csv_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.csv")
json_out = os.path.join(RESULTS_DIR, "transformer_experiments_summary.json")
df.to_csv(csv_out, index=False, encoding="utf-8")
with open(json_out, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

print(f"✓ Đã xuất bảng tổng hợp ({len(rows)}/7 runs) ra:\n  - {csv_out}\n  - {json_out}\n")
print(df.to_markdown(index=False))